In [ ]:
# ====================================================
# AI-BASED CUSTOMER CHURN PREDICTION USING ANN
# ====================================================

# Import Libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input


# ====================================================
# STEP 1 : CREATE DATASET
# ====================================================

np.random.seed(42)

# Create 1000 customer records

tenure = np.random.randint(1, 73, 1000)

monthly_charges = np.random.randint(
    20, 150, 1000
)

data_usage = np.random.uniform(
    1, 100, 1000
)

support_tickets = np.random.randint(
    0, 11, 1000
)


# ====================================================
# CREATE CHURN TARGET
# ====================================================

# Create a churn score based on customer behavior

churn_score = (
    -0.05 * tenure
    + 0.03 * monthly_charges
    - 0.01 * data_usage
    + 0.35 * support_tickets
)


# Add random noise

churn_score = (
    churn_score
    + np.random.normal(0, 1.5, 1000)
)


# Convert score into 0 or 1

# 0 = Customer Stays
# 1 = Customer Leaves

churn = (
    churn_score > 2.5
).astype(int)


# ====================================================
# CREATE DATAFRAME
# ====================================================

df = pd.DataFrame({

    "Tenure": tenure,

    "MonthlyCharges": monthly_charges,

    "DataUsage": data_usage,

    "SupportTickets": support_tickets,

    "Churn": churn

})


# Display first 5 records

print("\n================ DATASET PREVIEW ================\n")

print(df.head())


# Display dataset information

print("\n================ DATASET INFORMATION ================\n")

print(df.info())


# Display churn distribution

print("\n================ CHURN DISTRIBUTION ================\n")

print(df["Churn"].value_counts())


# ====================================================
# STEP 2 : INPUTS AND OUTPUT
# ====================================================

# Input Features

X = df[
    [
        "Tenure",
        "MonthlyCharges",
        "DataUsage",
        "SupportTickets"
    ]
]


# Target Variable

y = df["Churn"]


# ====================================================
# STEP 3 : TRAIN TEST SPLIT
# ====================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.2,

    random_state=42

)


print("\nTraining Samples :", len(X_train))

print("Testing Samples  :", len(X_test))


# ====================================================
# STEP 4 : FEATURE SCALING
# ====================================================

feature_scaler = StandardScaler()


# Fit scaler only on training data

X_train = feature_scaler.fit_transform(
    X_train
)


# Transform test data using the same scaler

X_test = feature_scaler.transform(
    X_test
)


# ====================================================
# STEP 5 : BUILD ANN MODEL
# ====================================================

model = Sequential([


    # Input Layer

    Input(shape=(4,)),


    # Hidden Layer 1

    Dense(
        16,
        activation="relu"
    ),


    # Hidden Layer 2

    Dense(
        8,
        activation="relu"
    ),


    # Output Layer

    Dense(
        1,
        activation="sigmoid"
    )

])


# ====================================================
# STEP 6 : COMPILE MODEL
# ====================================================

model.compile(

    optimizer="adam",

    loss="binary_crossentropy",

    metrics=["accuracy"]

)


# ====================================================
# STEP 7 : MODEL SUMMARY
# ====================================================

print("\n================ MODEL SUMMARY ================\n")

model.summary()


# ====================================================
# STEP 8 : TRAIN MODEL
# ====================================================

history = model.fit(

    X_train,

    y_train,

    epochs=100,

    batch_size=32,

    validation_split=0.2,

    verbose=1

)


# ====================================================
# STEP 9 : EVALUATE MODEL
# ====================================================

loss, accuracy = model.evaluate(

    X_test,

    y_test,

    verbose=0

)


print("\n================ MODEL EVALUATION ================\n")

print("Test Loss     :", loss)

print(
    "Test Accuracy :",
    accuracy
)

print(
    "Accuracy Percentage :",
    round(accuracy * 100, 2),
    "%"
)


# ====================================================
# STEP 10 : PREDICT NEW CUSTOMER CHURN
# ====================================================

# Create new customer as DataFrame
# This avoids the StandardScaler feature-name warning

new_customer = pd.DataFrame({

    "Tenure": [12],

    "MonthlyCharges": [120],

    "DataUsage": [20],

    "SupportTickets": [7]

})


print("\n================ NEW CUSTOMER ================\n")

print(new_customer)


# ====================================================
# SCALE NEW CUSTOMER
# ====================================================

# IMPORTANT:
# Use transform(), NOT fit_transform()

new_customer_scaled = feature_scaler.transform(

    new_customer

)


# ====================================================
# PREDICT CHURN PROBABILITY
# ====================================================

churn_probability = model.predict(

    new_customer_scaled,

    verbose=0

)


# Display probability

probability = (
    churn_probability[0][0] * 100
)


print(
    "\nChurn Probability :",
    round(probability, 2),
    "%"
)


# ====================================================
# CONVERT PROBABILITY INTO 0 OR 1
# ====================================================

churn_prediction = (

    churn_probability[0][0] >= 0.5

)


# ====================================================
# DISPLAY FINAL PREDICTION
# ====================================================

if churn_prediction:

    print(
        "Prediction : Customer is likely to CHURN."
    )

else:

    print(
        "Prediction : Customer is likely to STAY."
    )


# ====================================================
# STEP 11 : TRAINING ACCURACY GRAPH
# ====================================================

plt.figure(figsize=(8, 5))


plt.plot(

    history.history["accuracy"],

    label="Training Accuracy"

)


plt.plot(

    history.history["val_accuracy"],

    label="Validation Accuracy"

)


plt.title(
    "Training Accuracy vs Validation Accuracy"
)


plt.xlabel("Epoch")


plt.ylabel("Accuracy")


plt.legend()


plt.grid()


plt.show()


# ====================================================
# STEP 12 : TRAINING LOSS GRAPH
# ====================================================

plt.figure(figsize=(8, 5))


plt.plot(

    history.history["loss"],

    label="Training Loss"

)


plt.plot(

    history.history["val_loss"],

    label="Validation Loss"

)


plt.title(
    "Training Loss vs Validation Loss"
)


plt.xlabel("Epoch")


plt.ylabel("Loss")


plt.legend()


plt.grid()


plt.show()


# ====================================================
# PROJECT COMPLETED
# ====================================================

print("\n====================================================")

print("CUSTOMER CHURN PREDICTION USING ANN COMPLETED")

print("====================================================")


================ DATASET PREVIEW ================

   Tenure  MonthlyCharges  DataUsage  SupportTickets  Churn
0      52             112  17.316667               9      0
1      15              27  14.041149               2      0
2      72              84  72.872011               2      0
3      61              76  81.960748               6      0
4      21              86  22.137625               6      1

================ DATASET INFORMATION ================

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Tenure          1000 non-null   int32  
 1   MonthlyCharges  1000 non-null   int32  
 2   DataUsage       1000 non-null   float64
 3   SupportTickets  1000 non-null   int32  
 4   Churn           1000 non-null   int64  
dtypes: float64(1), int32(3), int64(1)
memory usage: 27.5 KB
None

================ CHURN DISTRIBUTION ================

Ch

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                      │ (None, 16)                  │              80 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, 8)                   │             136 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_8 (Dense)                      │ (None, 1)                   │               9 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 225 (900.00 B)

 Trainable params: 225 (900.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
